# 📝 과제 01: 타이타닉 생존 예측

## 만들 결과

day07 EDA에서는 타이타닉 승객의 생존 여부를 성별·객실 등급별 그래프로 살펴봤습니다. 이번에는 은닉층이 있는 신경망(DNN)이 승객 정보 6개만 보고 생존 여부를 맞히게 합니다. 교안 02에서 손글씨 숫자를 분류한 순서를 승객 표에 그대로 적용합니다.

<img src="images/40_titanic_pipeline.png" width="1050" alt="승객 정보 6개를 표준화해 미니배치로 DNN에 넣고 점수 2개 중 큰 쪽으로 사망과 생존을 예측하는 흐름">

- 승객 정보를 텐서로 바꾸고 학습 데이터의 평균·표준편차로 표준화합니다.
- DataLoader와 DNN을 만들고, 학습 전 첫 손실이 log 2 근처인지 점검합니다.
- 한 번 갱신해 보고, 에폭을 반복하는 학습 함수를 만듭니다.
- 학습률 후보 다섯 개를 학습 손실 곡선으로 비교해 하나를 고릅니다.
- 고른 학습률로 최종 모델을 학습해 테스트 정확도(약 80%)를 한 번 계산하고, 가상의 승객 두 명을 예측합니다.

## 준비: 데이터와 제공 코드

- **자료:** `data/과제01/titanic.csv`(891행, day07과 같은 seaborn titanic 데이터). 맞힐 값은 `survived`(1=생존, 0=사망)입니다.
- **입력 열 6개:** `pclass`(객실 등급 1~3), `sex`(여성=1, 남성=0으로 바꿈), `age`(나이), `sibsp`(함께 탄 형제·배우자 수), `parch`(함께 탄 부모·자녀 수), `fare`(요금)
- **순서:** 텐서와 표준화 -> DataLoader -> 모델과 첫 손실 -> 한 번 갱신 -> 학습 함수 -> 학습률 비교 -> 최종 모델과 테스트 정확도 -> 새 승객 예측
- **제공 코드:** 데이터 준비(나이가 빈 행 제거, 학습 80%와 테스트 20% 분할), 교안 02의 `train_one_epoch`, `evaluate`, 결과 그래프
- **작성:** `[작성]` 셀의 `...`를 채우고 `[자가채점]` 셀을 실행하세요. 뒤 문항이 앞 문항의 변수를 이어 쓰므로 1번부터 차례로 풉니다.

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# 타이타닉 승객 891명
titanic = pd.read_csv(data_dir / "과제01" / "titanic.csv")
print("행 수:", len(titanic))
display(titanic.head())

In [ ]:
# 입력으로 쓸 열 6개와 맞힐 열
feature_cols = ["pclass", "sex", "age", "sibsp", "parch", "fare"]
target_col = "survived"

# sex를 숫자로 바꿉니다: 여성 1, 남성 0
passengers = titanic.copy()
passengers["sex"] = (passengers["sex"] == "female").astype(int)

# 나이가 비어 있는 행은 뺍니다
passengers = passengers.dropna(subset=["age"]).reset_index(drop=True)

# 학습 80%, 테스트 20%: 섞는 순서를 seed 0으로 고정해 누구나 같은 분할을 얻습니다
generator = torch.Generator().manual_seed(0)
order = torch.randperm(len(passengers), generator=generator).tolist()
n_train = int(len(passengers) * 0.8)
train_df = passengers.iloc[order[:n_train]]
test_df = passengers.iloc[order[n_train:]]

print("학습:", len(train_df), "명 | 테스트:", len(test_df), "명")
display(train_df[feature_cols + [target_col]].head())

In [ ]:
# 교안 02에서 만든 학습 함수와 평가 함수입니다
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    """미니배치마다 파라미터를 한 번씩 갱신하고, 그 에폭 미니배치 손실의 평균을 돌려줍니다."""
    total_loss = 0.0
    for X_batch, y_batch in loader:
        # 미니배치를 모델이 있는 계산 장치로 옮깁니다
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        # 스케줄러가 있으면 스텝마다 학습률을 바꿉니다
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    # len(loader)는 미니배치 수입니다
    return total_loss / len(loader)


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples

## 1. 승객 정보를 텐서로 바꾸고 표준화하기

**배경**: 요금(`fare`)은 0~512, 객실 등급(`pclass`)은 1~3처럼 열마다 값의 크기가 다르면 학습률 하나로 모든 가중치를 고르게 갱신하기 어렵습니다(교안 01의 입력 크기와 학습률). 그래서 열마다 표준화합니다. 평균과 표준편차는 **학습 데이터에서만** 계산하고, 테스트 데이터에도 그 값을 그대로 씁니다. 테스트 데이터는 학습에 쓰지 않은 새 데이터라는 가정이기 때문입니다.

- 예: 학습 데이터의 나이 평균이 30, 표준편차가 15라면 20세는 (20 - 30) / 15 ≈ -0.67이 됩니다. 테스트 데이터의 20세도 같은 -0.67이 되어야 합니다.

**요구사항**:
- (1) `train_df`, `test_df`의 `feature_cols` 열을 float32 텐서로, `survived` 열을 정수(`torch.long`) 텐서 **`y_train`**, **`y_test`** 로 만듭니다. `nn.CrossEntropyLoss`는 정답을 클래스 번호(정수)로 받습니다.
- (2) 학습 데이터 텐서의 열별 평균을 **`train_mean`**, 열별 표준편차를 **`train_std`** 에 담고(둘 다 모양 `(6,)`), 학습 데이터와 테스트 데이터를 **둘 다 이 값으로** 표준화해 **`X_train`**, **`X_test`** 에 담습니다.

**확인 기준**: `X_train`, `X_test`는 float32이고, `X_train`은 열마다 평균 0, 표준편차 1 근처이며, `X_test`에 `train_std`를 곱하고 `train_mean`을 더하면 `test_df`의 원래 값으로 돌아옵니다.

<details><summary>힌트</summary>

```text
접근방법:
- 표 -> 텐서 -> 열별 평균과 표준편차 -> (값 - 평균) / 표준편차

세부구현:
1. DataFrame에서 열을 골라 to_numpy()로 꺼내고, torch.tensor에 dtype=torch.float32(정답은 torch.long)를 줍니다.
2. 열마다 하나씩 구하려면 mean, std에 dim=0을 줍니다(행 방향으로 모읍니다).
3. 테스트 텐서에서도 학습 데이터의 평균을 빼고 학습 데이터의 표준편차로 나눕니다.
```

</details>

In [ ]:
# [작성] (1) 입력과 정답을 텐서로 바꾸세요.

# 입력 6열을 float32 텐서로
X_train_raw = ...
X_test_raw = ...

# 정답(생존 1, 사망 0)을 정수 텐서로
y_train = ...
y_test = ...

print("X_train_raw:", X_train_raw.shape, "| y_train:", y_train.shape)

In [ ]:
# [작성] (2) 학습 데이터의 평균과 표준편차로 두 데이터를 표준화하세요.

# 학습 데이터의 열별 평균과 표준편차
train_mean = ...
train_std = ...

# 두 데이터를 같은 평균, 표준편차로 표준화
X_train = ...
X_test = ...

# 표준화한 뒤 열별 평균 비교
mean_table = pd.DataFrame({"학습 데이터 평균": X_train.mean(dim=0).tolist(), "테스트 데이터 평균": X_test.mean(dim=0).tolist()}, index=feature_cols)
display(mean_table.round(3))

In [ ]:
# [자가채점] 텐서의 모양과 자료형, 표준화에 쓴 평균과 표준편차를 확인합니다.
for name, value in [("X_train", X_train), ("X_test", X_test), ("y_train", y_train), ("y_test", y_test), ("train_mean", train_mean), ("train_std", train_std)]:
    assert isinstance(value, torch.Tensor), f"{name}이 텐서가 아닙니다. `...`를 채웠는지 확인하세요."

assert X_train.shape == (len(train_df), 6), f"X_train의 모양은 (학습 승객 수 {len(train_df)}, 6)입니다. feature_cols 열만 골랐는지 확인하세요."
assert X_test.shape == (len(test_df), 6), f"X_test의 모양은 (테스트 승객 수 {len(test_df)}, 6)입니다."
assert X_train.dtype == torch.float32 and X_test.dtype == torch.float32, "X_train, X_test는 float32여야 nn.Linear에 넣을 수 있습니다. torch.tensor(..., dtype=torch.float32)로 만드세요."
assert y_train.dtype == torch.long and y_test.dtype == torch.long, "y_train, y_test는 정수(torch.long) 텐서여야 합니다. CrossEntropyLoss는 정답을 클래스 번호로 받습니다."
assert y_train.shape == (len(train_df),) and y_test.shape == (len(test_df),), "y_train, y_test는 승객 수만큼의 1차원 텐서입니다. survived 열 하나만 고르세요."
assert (y_train.cpu().numpy() == train_df["survived"].to_numpy()).sum() == len(train_df), "y_train은 train_df의 survived 열을 순서 그대로 담습니다."
assert (y_test.cpu().numpy() == test_df["survived"].to_numpy()).sum() == len(test_df), "y_test는 test_df의 survived 열을 순서 그대로 담습니다."

# 평균과 표준편차는 학습 데이터에서만 계산해야 합니다
assert train_mean.shape == (6,) and train_std.shape == (6,), "train_mean, train_std는 열마다 하나씩, 모양 (6,)입니다. mean, std에 dim=0을 주세요."
pandas_mean = train_df[feature_cols].mean().to_numpy()
assert abs(train_mean.cpu().numpy() - pandas_mean).max() < 1e-3 * abs(pandas_mean).max(), "train_mean이 학습 데이터(train_df)의 열별 평균과 다릅니다. 전체 데이터나 테스트 데이터로 계산하지 않았는지 확인하세요."
pandas_std = train_df[feature_cols].std().to_numpy()
assert (abs(train_std.cpu().numpy() / pandas_std - 1) < 0.01).sum() == 6, "train_std가 학습 데이터(train_df)의 열별 표준편차와 다릅니다."

# 표준화 결과: 학습 데이터는 평균 0, 표준편차 1 근처
assert X_train.mean(dim=0).abs().max().item() < 1e-3, "X_train의 열별 평균이 0이 아닙니다. 학습 데이터에서 train_mean을 뺐는지 확인하세요."
assert (X_train.std(dim=0) - 1).abs().max().item() < 0.01, "X_train의 열별 표준편차가 1이 아닙니다. train_std로 나눴는지 확인하세요."

# 테스트 데이터를 되돌리면 원래 값이어야 합니다(테스트 데이터의 평균으로 표준화하면 여기서 걸립니다)
restored = (X_test * train_std + train_mean).cpu().numpy()
assert abs(restored - test_df[feature_cols].to_numpy()).max() < 0.01, "X_test를 학습 데이터의 train_mean, train_std로 표준화했는지 확인하세요. 테스트 데이터의 평균과 표준편차를 쓰면 안 됩니다."
print("✅ 1번 통과")

학습 데이터의 열별 평균은 모두 0인데(-0.0은 0보다 아주 조금 작은 값을 반올림한 것으로 0과 같습니다) 테스트 데이터는 0에서 조금씩 벗어납니다. 학습 데이터의 평균으로 표준화했기 때문이고, 이것이 맞는 결과입니다.

## 2. 미니배치를 나눠 주는 DataLoader 만들기

**배경**: 학습 승객 전체로 한 번 갱신하는 대신, 64명씩 미니배치로 나눠 한 에폭 동안 여러 번 갱신합니다. 학습용은 에폭마다 순서를 섞고, 테스트용은 정확도만 계산하므로 섞지 않습니다.

- 예: 200명을 64명씩 나누면 64, 64, 64, 8명의 미니배치 4개가 되어 한 에폭은 4스텝입니다.

**요구사항**:
- `X_train`, `y_train`을 `TensorDataset`으로 묶어 미니배치 크기 64, 에폭마다 순서를 섞는 **`train_loader`** 를 만듭니다.
- `X_test`, `y_test`로 미니배치 크기 64, 순서를 섞지 않는 **`test_loader`** 를 만듭니다.
- 한 에폭의 스텝 수를 `math.ceil`로 계산해 **`steps_per_epoch`** 에 담습니다.

**확인 기준**: `steps_per_epoch`가 `len(train_loader)`와 같고, 첫 미니배치의 입력 모양이 `(64, 6)`입니다.

<details><summary>힌트</summary>

```text
접근방법:
- TensorDataset으로 입력과 정답을 한 쌍씩 묶고, DataLoader에 미니배치 크기와 섞을지를 줍니다.

세부구현:
1. DataLoader에 batch_size=64와 shuffle을 줍니다(학습용 True, 테스트용 False).
2. 스텝 수는 학습 승객 수를 64로 나눠 올림합니다.
```

</details>

In [ ]:
# [작성] 미니배치 64명씩 나눠 주는 DataLoader를 만드세요.
import math

from torch.utils.data import DataLoader, TensorDataset

# 학습용: 에폭마다 순서를 섞습니다
train_loader = ...

# 테스트용: 섞지 않습니다
test_loader = ...

# 한 에폭의 스텝 수
steps_per_epoch = ...
print("한 에폭의 스텝 수:", steps_per_epoch)

In [ ]:
# [자가채점] 미니배치 크기, 섞기 여부, 스텝 수를 확인합니다.
from torch.utils.data import RandomSampler, SequentialSampler

assert isinstance(train_loader, DataLoader) and isinstance(test_loader, DataLoader), "train_loader, test_loader는 DataLoader로 만드세요. `...`를 채웠는지 확인하세요."
assert len(train_loader.dataset) == len(X_train), "train_loader에는 X_train, y_train을 TensorDataset으로 묶어 넣습니다."
assert len(test_loader.dataset) == len(X_test), "test_loader에는 X_test, y_test를 TensorDataset으로 묶어 넣습니다."
assert train_loader.batch_size == 64 and test_loader.batch_size == 64, "두 DataLoader 모두 batch_size=64입니다."
assert isinstance(train_loader.sampler, RandomSampler), "train_loader는 에폭마다 순서를 섞어야 합니다. shuffle=True를 주세요."
assert isinstance(test_loader.sampler, SequentialSampler), "test_loader는 섞지 않습니다. shuffle=False로 두세요."

# 첫 미니배치의 모양
X_batch, y_batch = next(iter(train_loader))
assert X_batch.shape == (64, 6) and y_batch.shape == (64,), "미니배치 하나는 입력 (64, 6), 정답 (64,)이어야 합니다. TensorDataset에 입력과 정답을 이 순서로 넣었는지 확인하세요."

assert isinstance(steps_per_epoch, int), "steps_per_epoch는 정수입니다. math.ceil의 결과를 담으세요."
assert steps_per_epoch == len(train_loader), f"steps_per_epoch({steps_per_epoch})가 len(train_loader)({len(train_loader)})와 다릅니다. 학습 승객 수를 64로 나눠 올림했는지 확인하세요."
print("✅ 2번 통과")

## 3. DNN을 만들고 학습 전 첫 손실 점검하기

**배경**: 학습 전 모델은 사망과 생존 두 클래스에 거의 같은 점수(로짓)를 줍니다. 그래서 교차 엔트로피 손실이 -log(1/2) = log 2 ≈ 0.693 근처에서 시작합니다. 첫 손실이 이 값에서 크게 벗어나면 출력 개수나 정답 형식이 잘못된 것이므로 학습 전에 먼저 확인합니다(교안 02의 log 10 = 2.30 점검과 같은 방법).

**요구사항**:
- (1) 부를 때마다 새 모델을 돌려주는 함수 **`make_model()`** 을 만듭니다.
    - 모델을 만들기 직전에 `torch.manual_seed(0)`을 부릅니다. 그러면 언제 불러도 같은 초기 파라미터로 시작하고, 그 뒤 DataLoader가 섞는 순서도 같아져 결과가 재현됩니다.
    - `nn.Sequential`로 `입력 6 -> 은닉 유닛 32 -> ReLU -> 은닉 유닛 16 -> ReLU -> 출력 2` 모델을 만들어 계산 장치(`device`)에 올려 돌려줍니다.
- (2) `make_model()`이 돌려준 모델을 **`model`** 에, `nn.CrossEntropyLoss()`를 **`loss_fn`** 에 담고, 학습 전 `model`에 `X_train` 전체를 넣어 계산한 손실을 파이썬 실수로 **`first_loss`** 에 담습니다.

**확인 기준**: 층 구성이 위와 같고, `make_model()`을 두 번 불러도 첫 층의 가중치가 같고, `first_loss`가 log 2와 0.05 이내로 가깝습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 nn.Sequential 모델에서 층의 입력, 출력 크기만 바꿉니다.

세부구현:
1. nn.Linear(입력 수, 출력 수)와 nn.ReLU()를 차례로 nn.Sequential에 넣고, 끝에 .to(device)를 붙여 return 합니다.
2. 입력도 모델과 같은 장치에 있어야 하므로 X_train, y_train에 .to(device)를 붙여 넣습니다.
3. 손실 텐서에서 파이썬 실수를 꺼낼 때는 .item()을 씁니다.
4. Target 1 is out of bounds 오류가 나면 마지막 층의 출력 개수가 클래스 수(2)보다 적은 것입니다.
```

</details>

In [ ]:
# [작성] (1) 부를 때마다 같은 초기 파라미터로 새 모델을 만드는 함수를 완성하세요.
import torch.nn as nn


def make_model():
    """입력 6, 은닉 유닛 32와 16, 출력 2인 DNN을 새로 만들어 돌려줍니다."""
    ...


print(make_model())

In [ ]:
# [작성] (2) 모델과 손실함수를 준비하고 학습 전 첫 손실을 계산하세요.

# 모델과 손실함수
model = ...
loss_fn = ...

# 학습 전 첫 손실
first_loss = ...
print("첫 손실:", round(first_loss, 4), "| log 2:", round(math.log(2), 4))

In [ ]:
# [자가채점] 층 구성, 같은 초기 파라미터, 첫 손실을 확인합니다.
model_a = make_model()
assert isinstance(model_a, nn.Sequential), "make_model()은 nn.Sequential 모델을 return 해야 합니다. 함수 끝에 return을 썼는지 확인하세요."
assert len(model_a) == 5, "층은 Linear, ReLU, Linear, ReLU, Linear 다섯 개입니다."

# 층마다 종류와 크기
assert isinstance(model_a[0], nn.Linear) and model_a[0].in_features == 6 and model_a[0].out_features == 32, "첫 층은 입력 6, 출력 32인 nn.Linear입니다."
assert isinstance(model_a[1], nn.ReLU) and isinstance(model_a[3], nn.ReLU), "두 Linear 사이마다 nn.ReLU()를 넣으세요. ReLU가 없으면 층을 쌓아도 직선 하나와 같습니다."
assert isinstance(model_a[2], nn.Linear) and model_a[2].in_features == 32 and model_a[2].out_features == 16, "가운데 층은 입력 32, 출력 16인 nn.Linear입니다."
assert isinstance(model_a[4], nn.Linear) and model_a[4].in_features == 16, "마지막 층은 입력 16인 nn.Linear입니다."
assert model_a[4].out_features == 2, "마지막 층의 출력은 클래스 수(사망, 생존)와 같은 2개입니다."
assert next(model_a.parameters()).device.type == device.type, "모델을 계산 장치에 올리세요. 함수 안에서 .to(device)를 붙입니다."

# 부를 때마다 같은 초기 파라미터
model_b = make_model()
assert torch.equal(model_a[0].weight, model_b[0].weight), "make_model()을 두 번 부르면 첫 층 가중치가 같아야 합니다. 함수 안에서 모델을 만들기 직전에 torch.manual_seed(0)을 부르세요."

assert isinstance(model, nn.Sequential) and torch.equal(model[0].weight, model_a[0].weight), "model에는 make_model()이 돌려준, 아직 학습하지 않은 모델을 담으세요."
assert isinstance(loss_fn, nn.CrossEntropyLoss), "loss_fn은 nn.CrossEntropyLoss()입니다."

# 첫 손실
assert isinstance(first_loss, float), "first_loss는 파이썬 실수입니다. 손실 텐서에 .item()을 붙이세요."
assert abs(first_loss - math.log(2)) < 0.05, f"첫 손실 {first_loss:.4f}가 log 2(0.693)에서 멉니다. 학습 전 모델에 X_train 전체를 넣었는지, 정답으로 y_train을 넣었는지 확인하세요."
print("✅ 3번 통과")

## 4. 한 번 갱신하고 손실 다시 계산하기

**배경**: 학습은 예측 -> 손실 -> 기울기 -> 갱신의 반복입니다. 반복문을 만들기 전에 학습 데이터 전체로 딱 한 번 갱신해, 옵티마이저의 세 줄(`zero_grad`, `backward`, `step`)이 손실을 실제로 줄이는지 확인합니다.

**요구사항**:
- `model`의 파라미터를 학습률 0.001로 갱신하는 Adam 옵티마이저를 **`optimizer`** 에 담습니다.
- 기울기 비우기 -> `X_train` 전체로 손실 계산 -> 역전파 -> 갱신을 **한 번** 실행합니다.
- 갱신한 `model`로 손실을 다시 계산해 파이썬 실수로 **`loss_after_step`** 에 담습니다.

**확인 기준**: `loss_after_step`이 지금 `model`의 손실과 같고 `first_loss`보다 작습니다. 한 번 갱신이라 줄어든 양은 0.05보다 작습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 제공 함수 train_one_epoch 안의 네 줄을 미니배치 대신 학습 데이터 전체로 한 번만 실행합니다.

세부구현:
1. torch.optim.Adam에 model.parameters()와 lr=0.001을 줍니다.
2. optimizer.zero_grad() -> 손실 계산 -> loss.backward() -> optimizer.step() 순서입니다.
3. 갱신이 끝난 뒤 model에 X_train을 다시 넣어 손실을 새로 계산합니다. 갱신 전에 계산한 손실 텐서를 다시 쓰면 값이 그대로입니다.
```

</details>

In [ ]:
# [작성] 학습 데이터 전체로 딱 한 번 갱신하고 손실을 다시 계산하세요.

# 옵티마이저
optimizer = ...

# 기울기 비우기 -> 손실 계산 -> 역전파 -> 갱신
...

# 갱신한 모델로 손실 다시 계산
loss_after_step = ...
print("갱신 전 손실:", round(first_loss, 4), "| 갱신 후 손실:", round(loss_after_step, 4))

In [ ]:
# [자가채점] 옵티마이저 설정과 한 번 갱신한 뒤의 손실을 확인합니다.
assert isinstance(optimizer, torch.optim.Adam), "optimizer는 torch.optim.Adam으로 만드세요. `...`를 채웠는지 확인하세요."
assert optimizer.param_groups[0]["lr"] == 0.001, "학습률은 0.001입니다. torch.optim.Adam(..., lr=0.001)"
assert optimizer.param_groups[0]["params"][0] is model[0].weight, "옵티마이저에 model.parameters()를 넘기세요. 다른 모델의 파라미터를 넘기면 model이 갱신되지 않습니다."

assert isinstance(loss_after_step, float), "loss_after_step은 파이썬 실수입니다. 손실 텐서에 .item()을 붙이세요."
current_loss = loss_fn(model(X_train.to(device)), y_train.to(device)).item()
assert abs(loss_after_step - current_loss) < 1e-5, "loss_after_step이 지금 model의 손실과 다릅니다. 갱신한 뒤 model에 X_train을 다시 넣어 계산하세요."
assert loss_after_step < first_loss, "손실이 줄지 않았습니다. optimizer.step()까지 실행했는지, loss.backward()를 step 전에 불렀는지 확인하세요."
assert first_loss - loss_after_step < 0.05, "손실이 한 번 갱신치고 너무 많이 줄었습니다. 갱신은 한 번만 실행하세요. 여러 번 실행했다면 3번 셀부터 다시 실행하세요."
print("✅ 4번 통과")

한 번 갱신으로는 손실이 0.01도 줄지 않습니다. 그래서 5번에서는 미니배치마다 갱신하는 에폭을 20번 반복합니다(9스텝 x 20에폭 = 180번 갱신).

## 5. 에폭을 반복하는 학습 함수 만들기

**배경**: 6번에서 학습률 다섯 개를, 7번에서 고른 학습률을 다시 학습하므로 "새 모델을 만들어 정해진 에폭만큼 학습"하는 일을 함수 하나로 만들어 둡니다. 교안 01의 `train_line(x, y, lr, steps)`가 손실 기록을 함께 돌려준 것처럼, 에폭별 학습 손실을 함께 돌려줍니다.

- 예: `train_model(0.001)`은 `(학습한 모델, [1에폭 손실, 2에폭 손실, ..., 20에폭 손실])`을 돌려줍니다.

**요구사항**:
- (1) 에폭 수 **`num_epochs`** 를 20으로 두고, 학습률 `lr`을 받는 함수 **`train_model(lr)`** 을 만듭니다.
    - 함수 안에서 `make_model()`로 새 모델을 만들고, 그 모델의 파라미터로 학습률 `lr`인 Adam 옵티마이저를 만듭니다.
    - `num_epochs`번 반복하며 제공 함수 `train_one_epoch`로 `train_loader`를 학습하고, 돌려받은 학습 손실을 리스트에 모읍니다.
    - `(모델, 에폭별 학습 손실 리스트)`를 돌려줍니다.
- (2) `train_model(0.001)`의 결과를 **`model`**, **`loss_history`** 에 담습니다. 이 문항에서는 테스트 데이터를 쓰지 않습니다.

**확인 기준**: `train_model`을 같은 학습률로 두 번 불러도 손실 기록이 같고(매번 새 모델에서 시작), `loss_history`가 20개이며 마지막 손실이 첫 에폭보다 작고 0.5 아래입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 새 모델 -> 옵티마이저 -> 에폭 반복(train_one_epoch 한 번씩) -> 모델과 손실 리스트 return

세부구현:
1. train_one_epoch(모델, train_loader, loss_fn, 옵티마이저)는 그 에폭의 평균 학습 손실을 돌려줍니다.
2. 결과 두 개는 return 모델, 손실리스트 처럼 쉼표로 이어 돌려주고, 받을 때도 쉼표로 두 이름에 나눠 담습니다.
3. type Linear doesn't define __round__ 오류가 나면 return 순서가 (손실 리스트, 모델)로 뒤바뀐 것입니다.
```

</details>

In [ ]:
# [작성] (1) 새 모델을 학습률 lr로 num_epochs만큼 학습하는 함수를 완성하세요.
num_epochs = 20


def train_model(lr):
    """make_model()로 새 모델을 만들어 학습률 lr의 Adam으로 num_epochs 에폭 학습하고, (모델, 에폭별 학습 손실 리스트)를 돌려줍니다."""
    # 새 모델과 옵티마이저
    ...

    # 에폭마다 한 번씩 학습하며 학습 손실 모으기
    epoch_losses = []
    ...

    return ...

In [ ]:
# [작성] (2) 학습률 0.001로 학습하세요.
model, loss_history = ...
print("1에폭 학습 손실:", round(loss_history[0], 4), f"| {num_epochs}에폭 학습 손실:", round(loss_history[-1], 4))

In [ ]:
# [자가채점] 함수가 매번 새 모델에서 시작하는지, 학습이 실제로 됐는지 확인합니다.
assert num_epochs == 20, "num_epochs는 20입니다."
assert isinstance(model, nn.Sequential), "model에는 train_model(0.001)이 돌려준 모델을 담으세요. return 모델, 손실리스트 순서인지 확인하세요."
assert isinstance(loss_history, list) and len(loss_history) == num_epochs, f"loss_history는 에폭마다 하나씩, {num_epochs}개인 리스트입니다. 반복문 안에서 train_one_epoch의 결과를 append 했는지 확인하세요."
assert isinstance(loss_history[0], float), "loss_history에는 train_one_epoch가 돌려준 파이썬 실수를 그대로 담습니다."
assert loss_history[-1] < loss_history[0] and loss_history[-1] < 0.5, f"마지막 에폭 학습 손실({loss_history[-1]:.3f})이 충분히 줄지 않았습니다. 에폭마다 train_one_epoch(모델, train_loader, loss_fn, 옵티마이저)를 불렀는지 확인하세요."

# 같은 학습률로 다시 부르면 같은 기록이 나와야 합니다(앞 모델을 이어서 학습하면 여기서 걸립니다)
_, again_history = train_model(0.001)
assert abs(again_history[0] - loss_history[0]) < 1e-4 and abs(again_history[-1] - loss_history[-1]) < 1e-4, "train_model을 다시 불렀더니 손실 기록이 달라졌습니다. 함수 안에서 make_model()로 새 모델을, 그 모델로 새 옵티마이저를 만드세요."
print("✅ 5번 통과")

In [ ]:
# 에폭별 학습 손실 곡선
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, num_epochs + 1), loss_history, marker="o", markersize=3, label="학습 손실(학습률 0.001)")

# 학습 전 손실 log 2를 기준선으로
ax.axhline(math.log(2), color="gray", linestyle="--", label="log 2 (학습 전)")
ax.set_xticks(range(0, num_epochs + 1, 5))
ax.set_xlabel("에폭")
ax.set_ylabel("학습 손실")
ax.legend(loc="lower left")
plt.show()

## 6. 학습률 후보를 학습 손실로 비교하기

**배경**: 교안 01에서 정한 순서(10배 간격으로 넓게 시도 -> 발산 경계 찾기 -> 정한 횟수 안에 수렴한 후보 가운데 발산 경계에서 가장 먼 값)로 학습률을 고릅니다.

- `train_model`은 매번 같은 초기 파라미터에서 시작하므로 학습률만 다른 공정한 비교가 됩니다. 비교는 **학습 손실**로만 하고, 테스트 데이터는 7번에서 마지막에 한 번만 씁니다.
- **수렴한 후보**는 20에폭 뒤 마지막 학습 손실이 **0.45 아래**인 후보로 정합니다. 학습 전 손실이 log 2(0.69)이므로 0.45 아래면 손실이 충분히 내려온 것입니다.
- 수렴한 후보는 모두 발산 경계보다 작으므로, 그 가운데 발산 경계에서 가장 먼 값은 **가장 작은 학습률**입니다. 마지막 학습 손실이 가장 낮은 값을 고르지 않는 이유는 경계에 가까울수록 데이터가 조금만 바뀌어도 발산할 수 있기 때문입니다.

**요구사항**:
- (1) `lr_candidates`의 학습률마다 `train_model(lr)`로 학습하고, `학습률`, `첫 에폭 학습 손실`, `마지막 학습 손실` 세 키의 dict를 이 순서로 리스트 **`lr_rows`** 에 더합니다. 에폭별 손실은 그래프용으로 `lr_curves`에 담겨 있습니다(제공 줄).
    - 예: `{"학습률": 0.0001, "첫 에폭 학습 손실": (손실 리스트의 첫 값), "마지막 학습 손실": (끝 값)}`
- (2) 표 `lr_table`에서 마지막 학습 손실이 0.45 아래인 행만 남긴 표를 **`converged_table`**(DataFrame, 열은 `lr_table`과 같음)에 담고, 그 가운데 가장 작은 학습률을 실수 **`best_lr`** 에 담습니다.

**확인 기준**: 후보 5개가 `lr_candidates` 순서대로 있고, `converged_table`에 마지막 학습 손실이 0.45 아래인 후보만 있으며, `best_lr`가 그 가운데 가장 작은 학습률입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 반복문 한 번에 train_model 한 번, lr_rows에 한 줄
- 표 -> 수렴한 행만 남기기 -> 남은 행에서 가장 작은 학습률

세부구현:
1. 손실 리스트의 첫 값은 [0], 끝 값은 [-1]로 꺼냅니다.
2. 행을 조건으로 남길 때는 표[조건] 형태로 씁니다. 조건은 마지막 학습 손실 열 < 0.45 입니다.
3. 남은 표의 학습률 열에 min()을 붙이면 가장 작은 학습률이 나옵니다.
```

</details>

In [ ]:
# [작성] (1) 학습률 후보마다 학습해 첫 에폭과 마지막 에폭의 학습 손실을 모으세요.
lr_candidates = [0.0001, 0.001, 0.01, 0.1, 1.0]

lr_rows = []
lr_curves = {}
for lr in lr_candidates:
    # 학습률 lr로 새 모델 학습
    candidate_model, candidate_losses = ...

    # 에폭별 학습 손실은 그래프용으로 보관합니다
    lr_curves[lr] = candidate_losses

    # 학습률, 첫 에폭 학습 손실, 마지막 학습 손실
    lr_rows.append(...)

lr_table = pd.DataFrame(lr_rows)
display(lr_table.round({"첫 에폭 학습 손실": 3, "마지막 학습 손실": 3}))

In [ ]:
# 학습률별 에폭 학습 손실 곡선: 왼쪽은 전체, 오른쪽은 손실 0.3~0.8 구간 확대
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
for ax in axes:
    for lr in lr_candidates:
        ax.plot(range(1, num_epochs + 1), lr_curves[lr], marker="o", markersize=4, label=f"학습률 {lr}")
    ax.axhline(math.log(2), color="gray", linestyle="--", label="log 2 (학습 전)")
    ax.set_xticks(range(0, num_epochs + 1, 5))
    ax.set_xlabel("에폭", fontsize=14)
    ax.tick_params(labelsize=12)

# 학습률 1.0의 첫 에폭 손실이 다른 후보보다 수십 배 커서 왼쪽은 세로축을 로그 눈금으로 그립니다
axes[0].set_yscale("log")
axes[0].set_ylabel("학습 손실 (로그 눈금)", fontsize=14)
axes[0].set_title("전체", fontsize=15)
axes[1].set_ylim(0.3, 0.8)
axes[1].set_ylabel("학습 손실", fontsize=14)
axes[1].set_title("0.3~0.8 확대", fontsize=15)
axes[1].legend(loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# [작성] (2) 수렴한 후보 가운데 발산 경계에서 가장 먼 학습률을 고르세요.

# 마지막 학습 손실이 0.45 아래인 후보만 남깁니다
converged_table = ...
display(converged_table.round({"첫 에폭 학습 손실": 3, "마지막 학습 손실": 3}))

# 남은 후보 가운데 가장 작은 학습률
best_lr = ...
print("고른 학습률:", best_lr)

In [ ]:
# [자가채점] 후보별 기록과 고른 학습률을 확인합니다.
assert lr_candidates == [0.0001, 0.001, 0.01, 0.1, 1.0], "학습률 후보 목록(lr_candidates)은 바꾸지 마세요."
assert isinstance(lr_rows, list) and len(lr_rows) == len(lr_candidates), "후보마다 한 줄, 5줄을 lr_rows에 더하세요. `...`를 채웠는지 확인하세요."
last_losses = {}
for row, lr in zip(lr_rows, lr_candidates):
    assert list(row) == ["학습률", "첫 에폭 학습 손실", "마지막 학습 손실"], "lr_rows의 각 dict는 학습률, 첫 에폭 학습 손실, 마지막 학습 손실 세 키를 이 순서로 가집니다."
    assert row["학습률"] == lr, "lr_rows는 lr_candidates 순서대로 만드세요."
    assert abs(row["첫 에폭 학습 손실"] - lr_curves[lr][0]) < 1e-9 and abs(row["마지막 학습 손실"] - lr_curves[lr][-1]) < 1e-9, f"학습률 {lr}의 두 손실은 train_model이 돌려준 손실 리스트의 첫 값과 끝 값입니다."
    last_losses[lr] = row["마지막 학습 손실"]

# 학습률이 너무 작으면 느리고, 너무 크면 발산합니다
assert last_losses[0.0001] > last_losses[0.001] > last_losses[0.01], "학습률 0.0001, 0.001, 0.01 순서로 마지막 학습 손실이 작아져야 합니다. train_model(lr)에 반복문의 lr을 넘겼는지, 함수 안에서 옵티마이저에 lr=lr을 줬는지 확인하세요."
assert last_losses[1.0] > last_losses[0.01], "학습률 1.0 후보의 기록을 확인하세요. 옵티마이저에 lr=lr을 넘겼는지 확인하세요."

# 수렴한 후보: 마지막 학습 손실이 0.45 아래인 학습률
converged_lrs = []
for row in lr_rows:
    if row["마지막 학습 손실"] < 0.45:
        converged_lrs.append(row["학습률"])
assert isinstance(converged_table, pd.DataFrame), "converged_table은 lr_table에서 행을 골라 남긴 DataFrame입니다. `...`를 채웠는지 확인하세요."
assert converged_table["학습률"].tolist() == converged_lrs, f"converged_table의 학습률은 {converged_lrs}여야 합니다. lr_table에서 마지막 학습 손실이 0.45 아래인 행만 남기세요."

# best_lr: 수렴한 후보 가운데 발산 경계에서 가장 먼 값, 곧 가장 작은 학습률
assert best_lr in converged_lrs, f"best_lr({best_lr})는 수렴한 후보가 아닙니다. 마지막 학습 손실이 0.45 아래인 후보 {converged_lrs} 가운데서 고르세요."
assert best_lr == min(converged_lrs), f"best_lr({best_lr})가 수렴한 후보 {converged_lrs} 가운데 가장 작은 학습률이 아닙니다. 마지막 학습 손실이 가장 낮은 값이 아니라 발산 경계에서 가장 먼 값을 고릅니다. converged_table의 학습률 열에 min()을 붙이세요."
print("✅ 6번 통과")

학습률 0.0001은 20에폭 뒤에도 학습 손실이 0.652로 학습 전(log 2 = 0.69)과 거의 같습니다(느림). 학습률 1.0은 첫 에폭 손실이 69.313까지 커졌고(발산), 20에폭 뒤에도 0.759로 학습 전보다 높습니다. 발산 경계는 0.1과 1.0 사이에 있습니다.

마지막 학습 손실이 0.45 아래인 수렴한 후보는 0.001(0.399), 0.01(0.340), 0.1(0.373)입니다. 손실은 0.01이 가장 낮지만, 발산 경계에서 가장 먼 **0.001**을 고릅니다. 확대 그래프에서도 0.001은 손실이 에폭마다 꾸준히 줄고, 경계에 가까운 0.1은 에폭마다 오르내립니다.

## 7. 고른 학습률로 최종 모델을 학습하고 테스트 정확도 계산하기

**배경**: 학습률을 고른 뒤에야 학습에 쓰지 않은 테스트 데이터로 정확도를 **한 번** 계산합니다. 정확도 숫자 하나만으로는 쓸모를 판단하기 어려우므로, 아무것도 배우지 않고 모든 승객을 사망이라고 답할 때의 정확도와 비교합니다.

- 예: 테스트 승객 10명 중 6명이 사망이라면 모두 사망이라고 답해도 정확도는 6 / 10 = 0.6입니다.

**요구사항**:
- `train_model(best_lr)`로 학습한 결과를 **`model`**, **`loss_history`** 에 다시 담습니다.
- 제공 함수 `evaluate`로 `test_loader`의 손실과 정확도를 계산해 **`test_loss`**, **`test_acc`** 에 담습니다.
- 테스트 데이터에서 사망(0)인 승객의 비율을 파이썬 실수로 **`baseline_acc`** 에 담습니다.

**확인 기준**: `model`이 `best_lr`로 학습한 모델이고, `test_acc`가 0.70 이상이며 `baseline_acc`보다 높습니다.

<details><summary>힌트</summary>

```text
접근방법:
- 학습 한 번 -> 평가 한 번 -> 정답이 0인 비율

세부구현:
1. evaluate(모델, test_loader, loss_fn)은 (평균 손실, 정확도) 두 값을 돌려줍니다.
2. y_test == 0은 참과 거짓 텐서입니다. float()로 바꿔 mean()을 구하고 .item()으로 꺼내면 비율입니다.
```

</details>

In [ ]:
# [작성] 고른 학습률로 최종 모델을 학습하고 테스트 데이터로 한 번 평가하세요.

# 고른 학습률로 새 모델 학습
model, loss_history = ...

# 테스트 데이터로 한 번 평가
test_loss, test_acc = ...

# 모두 사망이라고 답할 때의 정확도
baseline_acc = ...
print("모델 테스트 정확도:", round(test_acc, 3), "| 모두 사망이라고 답할 때:", round(baseline_acc, 3))

In [ ]:
# [자가채점] 최종 모델, 테스트 정확도, 기준 정확도를 확인합니다.
best_last_loss = lr_table.loc[lr_table["학습률"] == best_lr, "마지막 학습 손실"].iloc[0]
assert isinstance(loss_history, list) and abs(loss_history[-1] - best_last_loss) < 1e-4, "model, loss_history는 train_model(best_lr)의 결과입니다. 6번에서 고른 best_lr로 학습했는지 확인하세요."

_, check_acc = evaluate(model, test_loader, loss_fn)
assert isinstance(test_acc, float) and abs(test_acc - check_acc) < 1e-9, "test_acc는 evaluate(model, test_loader, loss_fn)이 돌려준 두 번째 값(정확도)입니다."
assert test_acc >= 0.70, f"테스트 정확도 {test_acc:.3f}가 낮습니다. 1번부터 다시 실행해 보세요."

assert isinstance(baseline_acc, float), "baseline_acc는 파이썬 실수입니다. .item()을 붙이세요."
died_count = len(y_test) - y_test.sum().item()
assert abs(baseline_acc - died_count / len(y_test)) < 1e-6, "baseline_acc는 테스트 데이터(y_test)에서 사망(0)인 승객의 비율입니다. 생존(1)의 비율이 아닌지 확인하세요."
assert test_acc > baseline_acc, "모델 정확도가 모두 사망이라고 답할 때보다 높아야 합니다."
print("✅ 7번 통과")

In [ ]:
# 모두 사망이라고 답할 때와 모델의 테스트 정확도 비교
fig, ax = plt.subplots(figsize=(6, 4))
names = ["모두 사망이라고 답하기", f"DNN (학습률 {best_lr})"]
values = [baseline_acc, test_acc]
ax.bar(names, values, color=["lightgray", "tab:blue"])

# 막대 위에 정확도 숫자를 적습니다
for i in range(len(values)):
    ax.text(i, values[i] + 0.02, f"{values[i]:.3f}", ha="center")
ax.set_ylim(0, 1)
ax.set_ylabel("테스트 정확도")
plt.show()

모든 승객을 사망이라고 답해도 테스트 정확도는 0.594입니다. 학습률 0.001로 학습한 모델은 0.797로 약 20%p 높아, 승객 정보 6개에서 생존과 관련된 규칙을 배웠다는 것을 알 수 있습니다.

## 8. 새 승객 두 명의 생존 예측하기

**배경**: 새 데이터를 예측할 때도 입력을 **학습할 때와 같은 방법**으로 바꿔야 합니다. 새 승객을 표준화하지 않거나 새 승객끼리의 평균으로 표준화하면, 모델이 학습한 입력과 척도가 달라 예측이 틀어집니다.

| 승객 | `pclass` | `sex` | `age` | `sibsp` | `parch` | `fare` | 설명 |
|---|---|---|---|---|---|---|---|
| 가 | 3 | 0 | 20 | 0 | 0 | 8.05 | 3등석, 20세 남성, 혼자 탐, 요금 8.05 |
| 나 | 1 | 1 | 17 | 0 | 1 | 110.0 | 1등석, 17세 여성, 부모 한 명과 함께 탐, 요금 110.0 |

**요구사항**:
- 두 승객의 값을 표의 열 순서(`feature_cols` 순서)대로 float32 텐서로 만들고, **학습 데이터의 평균·표준편차(1번의 `train_mean`, `train_std`)로** 표준화해 **`new_X`**(모양 `(2, 6)`)에 담습니다.
- 7번의 `model`에 `new_X`를 넣어 승객마다 점수(로짓) 2개를 얻고, 점수가 큰 쪽의 번호를 파이썬 리스트로 **`new_pred`** 에 담습니다(0=사망, 1=생존).

**확인 기준**: `new_X`를 되돌리면 표의 값이고, `new_pred`가 지금 `model`의 점수에서 나온 예측이며, 두 승객의 예측이 서로 다릅니다.

<details><summary>힌트</summary>

```text
접근방법:
- 표 두 줄 -> float32 텐서 (2, 6) -> 학습 데이터 기준 표준화 -> 모델 -> 점수가 큰 쪽 번호

세부구현:
1. 승객 한 명을 리스트 하나로 적고, 두 리스트를 묶은 리스트를 torch.tensor에 넣습니다.
2. 점수가 큰 쪽 번호는 argmax(dim=1)로 구하고, .tolist()로 파이썬 리스트로 바꿉니다.
3. 모델에 넣을 때는 new_X에 .to(device)를 붙입니다.
```

</details>

In [ ]:
# [작성] 새 승객 두 명을 학습 데이터 기준으로 표준화하고 생존 여부를 예측하세요.

# 두 승객의 값(feature_cols 순서)을 float32 텐서로
new_raw = ...

# 학습 데이터의 평균, 표준편차로 표준화
new_X = ...

# 점수(로짓) 2개 중 큰 쪽의 번호
new_pred = ...
print("예측(0=사망, 1=생존):", new_pred)

In [ ]:
# [자가채점] 표준화한 입력과 예측을 확인합니다.
assert isinstance(new_X, torch.Tensor) and new_X.shape == (2, 6), "new_X는 모양 (2, 6)인 텐서입니다. 승객 두 명을 한 텐서로 묶으세요."
assert new_X.dtype == torch.float32, "new_X는 float32입니다. torch.tensor(..., dtype=torch.float32)"

# 학습 데이터의 평균, 표준편차로 되돌리면 표의 값이어야 합니다
table_values = torch.tensor([[3, 0, 20, 0, 0, 8.05], [1, 1, 17, 0, 1, 110.0]])
restored = (new_X.cpu() * train_std.cpu() + train_mean.cpu()).double()
assert (restored - table_values).abs().max().item() < 0.01, "new_X를 되돌렸을 때 표의 값이 아닙니다. 열 순서가 feature_cols와 같은지, 1번의 train_mean, train_std로 표준화했는지 확인하세요."

# 예측은 지금 model의 점수에서
assert isinstance(new_pred, list) and len(new_pred) == 2, "new_pred는 길이 2인 파이썬 리스트입니다. argmax 결과에 .tolist()를 붙이세요."
model_pred = model(new_X.to(device)).argmax(dim=1).tolist()
assert new_pred == model_pred, "new_pred가 7번 model의 예측과 다릅니다. new_X를 model에 넣어 점수가 큰 쪽(argmax(dim=1))을 고르세요."
assert new_pred[0] != new_pred[1], "두 승객의 예측이 같습니다. 5번 train_model 안에서 make_model()로 새 모델을 만들었는지, new_X를 1번의 train_mean, train_std로 표준화했는지 확인하세요."
print("✅ 8번 통과")

3등석 20세 남성(가)은 사망, 1등석 17세 여성(나)은 생존으로 예측합니다. day07 EDA에서 본 것처럼 여성과 1등석의 생존율이 높고 남성과 3등석의 생존율이 낮은 경향과 같습니다.